# FreshBasket Customer Churn: EDA and Quality Checks

Run `freshbasket-churn inspect` first, map the real source columns in `config/project.yaml`, and place the supplied tables in `data/raw/`. This notebook intentionally does not assume source-column names or fabricate sample results. The configured churn label covers April–June 2024; monthly model features are restricted to before April 2024.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "project.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from freshbasket_churn.config import load_config
from freshbasket_churn.data import inspect_tables, prepare_customer_dataset
from freshbasket_churn.quality import quality_checks

config_path = PROJECT_ROOT / "config" / "project.yaml"
config = load_config(config_path)
tables = inspect_tables(config, config_path)

In [ ]:
quality = quality_checks(tables, config)
for check_name, result in quality.items():
    print(f"\n{check_name}")
    display(result.head(20))

In [ ]:
customers, pre_window_activity = prepare_customer_dataset(tables, config)
print(f"Joined analysis customers: {len(customers):,}")
print(f"Observed churn-label rate: {customers['churn'].mean():.1%}")
display(customers.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cohort_rates = customers.groupby('signup_cohort', observed=True)['churn'].agg(['size', 'mean'])
cohort_rates.rename(columns={'size': 'customers', 'mean': 'churn_rate'}).plot(
    kind='bar', y='churn_rate', figsize=(10, 4), color='#168a70', legend=False
)
plt.title('Observed churn rate by signup cohort')
plt.ylabel('Churn rate')
plt.tight_layout()

Review the quality outputs before interpreting associations. Missing customer-months can reflect either incomplete data or legitimate signup timing; IQR outliers are review candidates, not automatic deletion instructions. Model validation and the explicitly assumption-based retention simulation are generated by `freshbasket-churn run`.